In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.formula.api import ols

### Difference-in-Differences (The Policy Change Analyzer)

**Difference-in-Differences (DiD)** is a method that compares the **change over time** between a **treatment group** and a **control group** to measure the causal effect of an intervention.

> **DiD answers:** *"What happened to the treatment group compared to what would have happened without the treatment?"*

It does this by looking at the **trend before and after** the intervention.

$$
\text{DiD} = (\text{Treatment}_{\text{After}} - \text{Treatment}_{\text{Before}}) - (\text{Control}_{\text{After}} - \text{Control}_{\text{Before}})
$$

#### Example: Minimum Wage Impact

| Group | Before | After | Change |
|-------|--------|-------|--------|
| **Treatment** | 10.0 | 12.5 | +2.5 |
| **Control** | 9.5 | 10.0 | +0.5 |
| **DiD** | | | **+2.0** |

**Interpretation:** The minimum wage increase caused a **+2.0** increase in employment.

| Step | Action |
|------|--------|
| 1 | Calculate the **change** in the treatment group (After − Before) |
| 2 | Calculate the **change** in the control group (After − Before) |
| 3 | The **difference between these changes** is the causal effect |

#### Real Business Examples of Difference-in-Differences

1. Uber Surge Pricing

| Component | Description |
|-----------|-------------|
| **Treatment** | Cities where surge pricing was introduced |
| **Control** | Cities without surge pricing |
| **Outcome** | Number of rides |
| **DiD** | Compare change in rides before/after in treatment vs control cities |

**Question:** Did surge pricing increase or decrease the number of rides?

2. Amazon Prime

| Component | Description |
|-----------|-------------|
| **Treatment** | Users who got Prime |
| **Control** | Users who didn't get Prime |
| **Outcome** | Annual spending |
| **DiD** | Compare spending change before/after Prime membership |

**Question:** Does Amazon Prime increase customer spending?

#### Step-by-Step
| Step | Action |
|------|--------|
| **1. Define** | Treatment group, Control group, Pre/Post periods |
| **2. Check** | Parallel trends assumption (visual + statistical) |
| **3. Calculate** | DiD = (Treatment_After - Treatment_Before) - (Control_After - Control_Before) |
| **4. Regress** | DiD as interaction term (Treatment × After) |
| **5. Robustness** | Sensitivity checks, placebo tests |

In [2]:
'''
                    ┌─────────────────────────────────────────────────────┐
                    │         DIFFERENCE-IN-DIFFERENCES                 │
                    │                                                   │
                    │  Treatment Group ────►  Control Group             │
                    │                                                   │
                    │         Before     │      After                   │
                    │  Treatment  ▲      │  Treatment  ▲                │
                    │  Group      │      │  Group      │                │
                    │  (Old)     ││      │  (New)     ││                │
                    │             │      │             │                │
                    │  Control   ││      │  Control   ││                │
                    │  Group     ││      │  Group     ││                │
                    │  (Old)    ││       │  (New)    ││                │
                    │             │      │             │                │
                    │   ─────────┴──────┴─────────────                 │
                    │         Time (Pre → Post)                         │
                    │                                                   │
                    │   DiD = (Treatment_After - Treatment_Before)      │
                    │        - (Control_After - Control_Before)         │
                    └─────────────────────────────────────────────────────┘

'''

'\n                    ┌─────────────────────────────────────────────────────┐\n                    │         DIFFERENCE-IN-DIFFERENCES                 │\n                    │                                                   │\n                    │  Treatment Group ────►  Control Group             │\n                    │                                                   │\n                    │         Before     │      After                   │\n                    │  Treatment  ▲      │  Treatment  ▲                │\n                    │  Group      │      │  Group      │                │\n                    │  (Old)     ││      │  (New)     ││                │\n                    │             │      │             │                │\n                    │  Control   ││      │  Control   ││                │\n                    │  Group     ││      │  Group     ││                │\n                    │  (Old)    ││       │  (New)    ││                │\n                    │

### Did a new website design increase sales?"

In [14]:
# Generate data
np.random.seed(42)
n = 1000
time_periods = 2  # Before and After
groups = ['Control', 'Treatment']

# Create data
data = []
for group in groups:
    for time in ['Before', 'After']:
        if group == 'Control':
            base_sales = 100  # Control group baseline
            time_effect = 5 if time == 'After' else 0  # Natural growth
            treatment_effect = 0
        else:  # Treatment
            base_sales = 100  # Same baseline
            time_effect = 5 if time == 'After' else 0  # Natural growth
            treatment_effect = 15 if time == 'After' else 0  # The intervention!
        
        # Add noise
        sales = np.random.normal(base_sales + time_effect + treatment_effect, 10, n//4)
        
        for sale in sales:
            data.append({
                'Group': group,
                'Time': time,
                'Sales': sale,
                'Is_Treatment': 1 if group == 'Treatment' else 0,
                'Is_After': 1 if time == 'After' else 0,
                'Interaction': 1 if (group == 'Treatment' and time == 'After') else 0
            })

df = pd.DataFrame(data)
df

,Group,Time,Sales,Is_Treatment,Is_After,Interaction
0,Control,Before,104.967142,0,0,0
1,Control,Before,98.617357,0,0,0
2,Control,Before,106.476885,0,0,0
3,Control,Before,115.230299,0,0,0
4,Control,Before,97.658466,0,0,0
...,...,...,...,...,...,...
995,Treatment,After,117.188997,1,1,1
996,Treatment,After,137.976865,1,1,1
997,Treatment,After,126.408429,1,1,1
998,Treatment,After,114.288210,1,1,1


In [7]:
# Calculate group means
group_means = df.groupby(['Group', 'Time'])['Sales'].mean().reset_index()

print("Group Means:")
print(group_means)


Group Means:
       Group    Time       Sales
0    Control   After  105.160989
1    Control  Before   99.975771
2  Treatment   After  121.309895
3  Treatment  Before   99.326627


In [13]:
treatment_before = group_means[(group_means['Group'] == 'Treatment') & (group_means['Time'] == 'Before')]['Sales'].values[0]
treatment_after = group_means[(group_means['Group'] == 'Treatment') & (group_means['Time'] == 'After')]['Sales'].values[0]
control_before = group_means[(group_means['Group'] == 'Control') & (group_means['Time'] == 'Before')]['Sales'].values[0]
control_after = group_means[(group_means['Group'] == 'Control') & (group_means['Time'] == 'After')]['Sales'].values[0]

did_effect = (treatment_after - treatment_before) - (control_after - control_before)
print(f"DiD Effect: {did_effect:.2f}")
print(f"Interpretation: The new website design increased sales by ${did_effect:.2f}")

DiD Effect: 16.80
Interpretation: The new website design increased sales by $16.80


### Regression Approach to DiD

In [15]:
# Sales = β₀ + β₁ * Treatment + β₂ * After + β₃ * (Treatment × After) + ε
model = ols('Sales ~ Is_Treatment + Is_After + Interaction', data=df).fit()
print(model.summary())

# Extract the DiD coefficient
did_coeff = model.params['Interaction']
print(f"\nDiD Effect from Regression: {did_coeff:.2f}")

# Predict counterfactual (what would have happened without treatment)
df['Predicted'] = model.predict(df)

# Visualize counterfactual
treatment_group = df[df['Is_Treatment'] == 1]
treatment_after = treatment_group[treatment_group['Is_After'] == 1]

# Average actual vs predicted for treatment group after intervention
actual_after = treatment_after['Sales'].mean()
predicted_after = treatment_after['Predicted'].mean()

print(f"\nTreatment Group After Intervention:")
print(f"  Actual Average Sales: {actual_after:.2f}")
print(f"  Predicted (Counterfactual): {predicted_after:.2f}")
print(f"  Causal Effect: {actual_after - predicted_after:.2f}")

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.453
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     274.5
Date:                Wed, 26 Aug 2026   Prob (F-statistic):          7.92e-130
Time:                        11:11:35   Log-Likelihood:                -3697.3
No. Observations:                1000   AIC:                             7403.
Df Residuals:                     996   BIC:                             7422.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       99.9758      0.619    161.620   